In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

import joblib

plt.style.use("ggplot")

In [ ]:
rfm = pd.read_csv(r"C:\Users\ka843\Coding\Elytespark_minor_project\data\processed\customer_rfm_clean.csv")

rfm.head()

In [ ]:
X = rfm[["Recency","Frequency","Monetary"]]

X.head()

In [ ]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

X_scaled = pd.DataFrame(
    X_scaled,
    columns=X.columns
)

X_scaled.head()

In [ ]:
joblib.dump(
    scaler,
    "../models/scaler.pkl"
)

print("Scaler Saved Successfully")

In [ ]:
X_scaled.describe().round(2)

In [ ]:
inertia = []

k_range = range(2,11)

for k in k_range:

    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    model.fit(X_scaled)

    inertia.append(model.inertia())

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(
    k_range,
    inertia,
    marker="o",
    linewidth=2
)

plt.xlabel("Number of Clusters (K)")

plt.ylabel("Inertia")

plt.title("Elbow Method")

plt.grid(True)

plt.savefig(
    "../reports/figures/elbow_method.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
scores = []

for k in range(2,11):

    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    labels = model.fit_predict(X_scaled)

    score = silhouette_score(
        X_scaled,
        labels
    )

    scores.append(score)

    print(f"K={k}  -->  {score:.4f}")

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(
    range(2,11),
    scores,
    marker="o",
    linewidth=2
)

plt.xlabel("Number of Clusters")

plt.ylabel("Silhouette Score")

plt.title("Silhouette Analysis")

plt.grid(True)

plt.savefig(
    "../reports/figures/silhouette_score.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
kmeans = KMeans(
    n_clusters=BEST_K,
    random_state=42,
    n_init=10
)

rfm["Cluster"] = kmeans.fit_predict(X_scaled)

rfm.head()

In [ ]:
joblib.dump(
    kmeans,
    "../models/kmeans_model.pkl"
)

print("KMeans Model Saved Successfully")

In [ ]:
rfm.to_csv(
    "../data/processed/customer_segments.csv",
    index=False
)

print("Customer Segments Saved Successfully")

In [ ]:
cluster_summary = rfm.groupby("Cluster").agg({
    "Recency": ["mean","median"],
    "Frequency": ["mean","median"],
    "Monetary": ["mean","median"],
    "CustomerID": "count"
})

cluster_summary.columns = [
    "Avg_Recency",
    "Median_Recency",
    "Avg_Frequency",
    "Median_Frequency",
    "Avg_Monetary",
    "Median_Monetary",
    "Customer_Count"
]

cluster_summary = cluster_summary.round(2)

cluster_summary

In [ ]:
cluster_summary.to_csv(
    "../outputs/cluster_statistics.csv"
)

cluster_summary